In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker('en_GB')
random.seed(42)

#------------------
# Create Properties
#------------------
num_properties = 100
properties = []

for i in range(1, num_properties + 1):
    build_year = random.randint(1960, 2023)

    # Randomly introduce missing inspection dates
    last_inspection = fake.date_between(start_date='-2y', end_date='today')
    if random.random() < 0.1:  # 10% missing
        last_inspection = None

    properties.append({
        'PropertyID': i,
        'Address': fake.address().replace('\n', ', '),
        'PropertyType': random.choice(['Flat', 'House']),
        'BuildYear': build_year,
        'Last_inspection_date': last_inspection.strftime('%Y-%m-%d') if last_inspection else None,
        'ConditionRating': random.randint(1, 5)
    })

df_properties = pd.DataFrame(properties)

#---------------
# Create Repairs
#---------------
num_repairs = 500
repairs = []

for i in range(1, num_repairs + 1):
    property_id = random.randint(1, num_properties)
    request_date = fake.date_between(start_date='-2y', end_date='today')

    priority = random.choice(['Routine', 'Urgent', 'Emergency'])

    # Completion date logic
    if priority == 'Emergency':
        completion_date = request_date + timedelta(days=random.randint(1, 3))
    else:
        completion_date = request_date + timedelta(days=random.randint(5, 15))

    # Introduce missing completion dates (e.g., still open)
    if random.random() < 0.05:  # 5% missing
        completion_date = None

    repair_type = random.choice(['Plumbing', 'Electrical', 'Heating', 'Roof'])

    base_cost = {
        'Plumbing': 100,
        'Electrical': 200,
        'Heating': 400,
        'Roof': 600
    }
    cost = base_cost[repair_type] + random.randint(-50, 200)

    repairs.append({
        'RepairID': i,
        'PropertyID': property_id,
        'RequestDate': request_date.strftime('%Y-%m-%d'),
        'CompletionDate': completion_date.strftime('%Y-%m-%d') if completion_date else None,
        'RepairType': repair_type,
        'Priority': priority,
        'Cost': cost
    })

df_properties_repairs = pd.DataFrame(repairs)

#-----------------
# Generate Tenants
#-----------------
tenants = []

for i in range(1, num_properties + 1):
    satisfaction = random.randint(1, 5)

    # Introduce missing satisfaction scores
    if random.random() < 0.05:
        satisfaction = None

    tenants.append({
        'TenantID': i,
        'PropertyID': i,
        'ComplaintFlag': 'Yes' if satisfaction and satisfaction <= 2 else 'No',
        'SatisfactionScore': satisfaction
    })

df_tenants = pd.DataFrame(tenants)

#------------
# Save to CSV
#------------

df_properties.to_csv('Properties.csv', index=False)
df_properties_repairs.to_csv('Repairs.csv', index=False)
df_tenants.to_csv('Tenants.csv', index=False)